# 🧬 Notebook 2 : La classification, pas à pas

**Objectif** : classer des échantillons **sains vs malades** à partir de mesures d'expression de gènes, et comprendre comment évaluer un classifieur.

## Plan
1. Les données (expression de 5 gènes, dont seulement 2 informatifs)
2. Entraînement / test (avec stratification)
3. Régression logistique et descente de gradient (loss = log-loss)
4. Régression logistique avec scikit-learn + frontière de décision
5. Les métriques : accuracy, matrice de confusion, précision, rappel, F1, ROC-AUC
6. Validation croisée stratifiée
7. Arbre de décision : sur-apprentissage
8. Forêt aléatoire et importance des gènes
9. Courbe d'apprentissage

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score,
                             classification_report)

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)   # reproductibilité

## 1. Les données

On simule **300 échantillons** (patients) avec l'expression de **5 gènes** :
- `gene_1` et `gene_2` sont **informatifs** : plus exprimés chez les malades ;
- `gene_3`, `gene_4`, `gene_5` sont du **bruit pur** (comme la majorité des gènes dans un vrai jeu de données !).

Étiquette (`label`) : **0 = sain**, **1 = malade** (environ 40 % de malades → classes **déséquilibrées**, courant en clinique).

In [ ]:
n = 300
label = (rng.random(n) < 0.40).astype(int)          # 1 = malade (≈ 40 %), 0 = sain

df = pd.DataFrame({
    "gene_1": rng.normal(5, 1.3, n) + 1.8 * label,   # informatif
    "gene_2": rng.normal(5, 1.3, n) + 1.5 * label,   # informatif
    "gene_3": rng.normal(5, 1.3, n),                 # bruit
    "gene_4": rng.normal(5, 1.3, n),                 # bruit
    "gene_5": rng.normal(5, 1.3, n),                 # bruit
    "label": label,
})
print(df["label"].value_counts().rename({0: "Sain", 1: "Malade"}))
df.head()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

# Nuage de points des 2 gènes informatifs
sns.scatterplot(data=df, x="gene_1", y="gene_2", hue="label", palette=["tab:blue", "tab:red"], ax=ax[0])
ax[0].set_title("Deux gènes informatifs")

# Distribution de chaque gène selon le groupe
df_long = df.melt(id_vars="label", var_name="gène", value_name="expression")
sns.boxplot(data=df_long, x="gène", y="expression", hue="label", palette=["tab:blue", "tab:red"], ax=ax[1])
ax[1].set_title("Expression par gène et par groupe")
plt.show()

## 2. Entraînement / test avec **stratification**

`stratify=y` garantit que la **proportion de malades** est la même dans le train et le test. Essentiel quand les classes sont déséquilibrées ou que les données sont rares.

In [ ]:
X = df.drop(columns="label")
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Train : {len(X_train)} (malades : {y_train.mean():.0%}) | Test : {len(X_test)} (malades : {y_test.mean():.0%})")

## 3. Régression logistique et descente de gradient

Malgré son nom, c'est un modèle de **classification**. Il calcule une probabilité :

```
z = w₁·gene_1 + w₂·gene_2 + b
p = σ(z) = 1 / (1 + e^(−z))          ← la sigmoïde écrase z entre 0 et 1
```

On prédit **« malade » si p ≥ 0,5**.

**Loss : la log-loss (entropie croisée binaire)**

```
Loss = − moyenne( y·log(p) + (1−y)·log(1−p) )
```

Elle punit **très fort** les prédictions confiantes mais fausses. Son gradient est élégant :
- ∂Loss/∂w = Xᵀ·(p − y) / n
- ∂Loss/∂b = moyenne(p − y)

In [ ]:
# Pour la démonstration, on utilise uniquement les 2 gènes informatifs, standardisés
scaler2 = StandardScaler().fit(X_train[["gene_1", "gene_2"]])
X2_tr = scaler2.transform(X_train[["gene_1", "gene_2"]])
y_tr = y_train.values

def sigmoide(z):
    return 1 / (1 + np.exp(-z))

def logistique_gd(X, y, lr=0.5, n_epochs=200):
    n, d = X.shape
    w, b = np.zeros(d), 0.0
    historique = []
    for epoch in range(n_epochs):
        p = sigmoide(X @ w + b)                                        # 1) probabilités prédites
        loss = -np.mean(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12))  # 2) log-loss (1e-12 évite log(0))
        historique.append(loss)
        grad_w = X.T @ (p - y) / n                                     # 3) gradients
        grad_b = np.mean(p - y)
        w -= lr * grad_w                                               # 4) mise à jour
        b -= lr * grad_b
    return w, b, historique

w, b, hist = logistique_gd(X2_tr, y_tr)
print(f"w = {np.round(w, 3)}, b = {b:.3f}, log-loss finale = {hist[-1]:.3f}")

plt.figure(figsize=(7, 4))
plt.plot(hist)
plt.xlabel("Epoch"); plt.ylabel("Log-loss")
plt.title("Convergence de la descente de gradient (logistique)")
plt.show()

## 4. Régression logistique avec scikit-learn + frontière de décision

On utilise un **pipeline** : standardisation puis modèle. Avantage : le scaler est appris sur le **train uniquement**, ce qui évite la **fuite de données (data leakage)**.

In [ ]:
# Modèle sur les 2 gènes informatifs, pour pouvoir visualiser la frontière en 2D
logreg2 = make_pipeline(StandardScaler(), LogisticRegression())
logreg2.fit(X_train[["gene_1", "gene_2"]], y_train)

# Grille de points couvrant le plan pour colorier les zones de décision
g1, g2 = np.meshgrid(np.linspace(1, 11, 300), np.linspace(1, 11, 300))
grille = pd.DataFrame({"gene_1": g1.ravel(), "gene_2": g2.ravel()})
proba = logreg2.predict_proba(grille)[:, 1].reshape(g1.shape)   # P(malade) en chaque point

plt.figure(figsize=(7.5, 5.5))
cs = plt.contourf(g1, g2, proba, levels=20, cmap="coolwarm", alpha=0.6)
plt.colorbar(cs, label="P(malade)")
plt.contour(g1, g2, proba, levels=[0.5], colors="k", linewidths=2)   # frontière de décision p = 0.5
plt.scatter(X_train["gene_1"], X_train["gene_2"], c=y_train, cmap="coolwarm", edgecolor="k", s=30)
plt.xlabel("gene_1"); plt.ylabel("gene_2")
plt.title("Frontière de décision (régression logistique)")
plt.show()

## 5. Les métriques de classification

On compare les prédictions aux vraies étiquettes. Quatre cas possibles (en considérant « malade » = positif) :

| | Prédit sain | Prédit malade |
|---|---|---|
| **Vraiment sain** | VN (vrai négatif) | **FP** (faux positif) |
| **Vraiment malade** | **FN** (faux négatif) | VP (vrai positif) |

| Métrique | Formule | Question posée |
|---|---|---|
| **Accuracy** | (VP+VN) / total | Quelle proportion de bonnes réponses ? |
| **Précision** | VP / (VP+FP) | Quand je dis « malade », ai-je raison ? |
| **Rappel (sensibilité)** | VP / (VP+FN) | Combien de vrais malades ai-je détectés ? |
| **F1** | moyenne harmonique précision/rappel | Compromis entre les deux |
| **ROC-AUC** | aire sous la courbe ROC | Capacité à **séparer** les classes, indépendamment du seuil |

⚠️ L'**accuracy peut tromper** quand les classes sont déséquilibrées : avec 99 % de sains, un modèle qui répond toujours « sain » a 99 % d'accuracy… mais ne détecte aucun malade. En diagnostic, on surveille souvent le **rappel** (rater un malade coûte cher).

In [ ]:
# Modèle sur les 5 gènes (le modèle doit se débrouiller avec les gènes de bruit)
logreg = make_pipeline(StandardScaler(), LogisticRegression())
logreg.fit(X_train, y_train)

y_pred = logreg.predict(X_test)                   # classe prédite (seuil 0.5)
y_proba = logreg.predict_proba(X_test)[:, 1]      # probabilité d'être malade

print(f"Accuracy  : {accuracy_score(y_test, y_pred):.3f}")
print(f"Précision : {precision_score(y_test, y_pred):.3f}")
print(f"Rappel    : {recall_score(y_test, y_pred):.3f}")
print(f"F1        : {f1_score(y_test, y_pred):.3f}")
print(f"ROC-AUC   : {roc_auc_score(y_test, y_proba):.3f}")
print()
print(classification_report(y_test, y_pred, target_names=["Sain", "Malade"]))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

# Matrice de confusion
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=["Sain", "Malade"]).plot(ax=ax[0], cmap="Blues")
ax[0].set_title("Matrice de confusion")
ax[0].grid(False)

# Courbe ROC : taux de vrais positifs vs taux de faux positifs pour tous les seuils possibles
fpr, tpr, seuils = roc_curve(y_test, y_proba)
ax[1].plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc_score(y_test, y_proba):.3f}")
ax[1].plot([0, 1], [0, 1], "k--", label="Hasard (AUC = 0.5)")
ax[1].set_xlabel("Taux de faux positifs (1 − spécificité)")
ax[1].set_ylabel("Taux de vrais positifs (rappel)")
ax[1].set_title("Courbe ROC"); ax[1].legend()
plt.show()

## 6. Validation croisée stratifiée

Même principe que pour la régression, mais chaque fold **conserve la proportion de classes**. Avec peu d'échantillons (fréquent en biologie), c'est bien plus fiable qu'un seul découpage.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for nom, scoring in [("Accuracy", "accuracy"), ("F1", "f1"), ("ROC-AUC", "roc_auc")]:
    scores = cross_val_score(logreg, X, y, cv=skf, scoring=scoring)
    print(f"{nom:9s} : {np.round(scores, 3)} → moyenne = {scores.mean():.3f} ± {scores.std():.3f}")

## 7. Arbre de décision : le sur-apprentissage en action

Un arbre pose une série de questions du type « `gene_1 > 6.2` ? ». Sans limite de profondeur, il peut **apprendre le train par cœur** (variance élevée).

On fait varier `max_depth` (la complexité) et on compare train vs test.

In [ ]:
profondeurs = range(1, 16)
acc_tr, acc_cv = [], []
for d in profondeurs:
    arbre = DecisionTreeClassifier(max_depth=d, random_state=42)
    arbre.fit(X_train, y_train)
    acc_tr.append(arbre.score(X_train, y_train))                                   # accuracy sur le train
    acc_cv.append(cross_val_score(arbre, X_train, y_train, cv=skf).mean())         # accuracy en CV

plt.figure(figsize=(8, 4.5))
plt.plot(profondeurs, acc_tr, "o-", label="Train")
plt.plot(profondeurs, acc_cv, "o-", label="Validation croisée")
plt.xlabel("Profondeur maximale de l'arbre"); plt.ylabel("Accuracy")
plt.title("Arbre de décision : sur-apprentissage quand la profondeur augmente")
plt.legend()
plt.show()
print("Train → ~100 % pour les arbres profonds, mais la validation stagne ou baisse : c'est de l'overfitting.")

## 8. Forêt aléatoire et importance des gènes

Une **forêt aléatoire** entraîne **beaucoup d'arbres** sur des sous-échantillons de données et de variables, puis fait **voter** les arbres. Ça **réduit la variance** et marche très bien « out of the box ».

Bonus : elle donne l'**importance de chaque variable** → utile pour identifier des **gènes candidats (biomarqueurs)**.

In [ ]:
foret = RandomForestClassifier(n_estimators=300, max_depth=5, random_state=42)
foret.fit(X_train, y_train)

y_pred_rf = foret.predict(X_test)
print(f"Forêt aléatoire → Accuracy test = {accuracy_score(y_test, y_pred_rf):.3f} | "
      f"F1 = {f1_score(y_test, y_pred_rf):.3f} | ROC-AUC = {roc_auc_score(y_test, foret.predict_proba(X_test)[:, 1]):.3f}")

# Importance des gènes : on s'attend à voir gene_1 et gene_2 en tête (les 3 autres sont du bruit)
importances = pd.Series(foret.feature_importances_, index=X.columns).sort_values()
plt.figure(figsize=(6, 3.5))
importances.plot(kind="barh", color="teal")
plt.xlabel("Importance"); plt.title("Importance des gènes (forêt aléatoire)")
plt.show()

## 9. Courbe d'apprentissage

Même lecture que dans le notebook de régression : on regarde l'évolution des scores selon le **nombre d'échantillons** d'entraînement.

In [ ]:
def tracer_courbe(modele, titre):
    tailles, sc_tr, sc_val = learning_curve(
        modele, X, y, cv=skf, scoring="accuracy",
        train_sizes=np.linspace(0.15, 1.0, 8),
    )
    plt.plot(tailles, sc_tr.mean(axis=1), "o-", label="Train")
    plt.plot(tailles, sc_val.mean(axis=1), "o-", label="Validation (CV)")
    plt.xlabel("Échantillons d'entraînement"); plt.ylabel("Accuracy")
    plt.title(titre); plt.ylim(0.5, 1.02); plt.legend()

plt.figure(figsize=(12, 4.5))
plt.subplot(1, 2, 1); tracer_courbe(logreg, "Régression logistique")
plt.subplot(1, 2, 2); tracer_courbe(DecisionTreeClassifier(random_state=42), "Arbre non limité (variance)")
plt.tight_layout(); plt.show()

## ✅ Récapitulatif

| Concept | À retenir |
|---|---|
| Stratification | Garder les proportions de classes dans chaque découpage |
| Log-loss | Loss de la classification, punit les erreurs confiantes |
| Sigmoïde + seuil | Transforme un score en probabilité puis en classe |
| Précision / Rappel / F1 | Mesures adaptées au déséquilibre de classes |
| ROC-AUC | Qualité de séparation indépendamment du seuil |
| Validation croisée stratifiée | Estimation fiable avec peu d'échantillons |
| Arbre profond | Risque d'overfitting (variance) |
| Forêt aléatoire | Réduit la variance, donne l'importance des gènes |
| Pipeline | Évite la fuite de données (scaler appris sur le train seulement) |

## 🧪 Exercices
1. Change le seuil de décision (0.3 puis 0.7) avec `y_proba >= seuil` : comment évoluent précision et rappel ?
2. Passe à 10 % de malades (`rng.random(n) < 0.10`) : que devient l'accuracy ? Et le rappel ?
3. Remplace les 3 gènes de bruit par 500 gènes de bruit : que se passe-t-il pour l'arbre, la forêt, la régression logistique ? (indice : essaie `LogisticRegression(penalty="l1", solver="liblinear")`)
4. Applique tout ce notebook à un vrai jeu de données : `from sklearn.datasets import load_breast_cancer`.